# 01 - FF-HEDM forward simulation and gradient-based recovery

End-to-end demo of `midas-diffract`:

1. Build an FF-HEDM geometry and forward model.
2. Generate a reflection list with `midas-hkls`.
3. Forward-simulate a known grain (orientation + lattice).
4. Verify gradients flow through the model.
5. Recover the grain from a perturbed start via the 3-phase L-BFGS schedule.

No C binaries required. The reflection list comes from the optional
`midas-hkls` companion package; for the pixel-exact C cross-validation
tests see `tests/test_c_comparison.py`.

**Setup:**
```bash
pip install midas-diffract[hkls]    # pulls in midas-hkls
```

In [ ]:
import math

import torch

import midas_diffract as md
from midas_hkls import Lattice, SpaceGroup

DEG2RAD = math.pi / 180.0
torch.set_default_dtype(torch.float64)

## Geometry

In [ ]:
geom = md.HEDMGeometry(
    Lsd=1_000_000.0,        # sample-to-detector (um)
    y_BC=1024.0, z_BC=1024.0,
    px=200.0,
    omega_start=0.0, omega_step=0.25, n_frames=1440,
    n_pixels_y=2048, n_pixels_z=2048,
    min_eta=6.0,
    wavelength=0.172979,    # Angstroms
)
geom

## Reflection list (Au, FCC, space group 225)

`midas_diffract.hkls_for_forward_model` wraps `midas_hkls.generate_hkls`,
expands every ASU representative to all symmetry-equivalent reflections,
and computes the reference Cartesian G-vectors. The B-matrix convention
matches the one in `HEDMForwardModel.correct_hkls_latc`, so the strain
path stays self-consistent.

In [ ]:
sg = SpaceGroup.from_number(225)                # FCC
lat = Lattice.for_system('cubic', a=4.08)         # Au

hkls_cart, thetas, hkls_int = md.hkls_for_forward_model(
    sg, lat,
    wavelength_A=geom.wavelength,
    two_theta_max_deg=15.0,
)
print(f'{hkls_cart.shape[0]} reflections, max 2theta = '
      f'{(2 * thetas.max().item()) / DEG2RAD:.2f} deg')

## Forward model

In [ ]:
model = md.HEDMForwardModel(
    hkls=hkls_cart,
    thetas=thetas,
    geometry=geom,
    hkls_int=hkls_int,
)

## Ground-truth forward simulation

In [ ]:
gt_euler = torch.tensor([45.0, 30.0, 60.0]) * DEG2RAD
gt_latc = torch.tensor([4.082, 4.079, 4.081, 90.01, 89.99, 90.02])
gt_pos = torch.zeros(3)

spots = model(gt_euler.unsqueeze(0), gt_pos.unsqueeze(0), lattice_params=gt_latc)
coords, valid = md.HEDMForwardModel.predict_spot_coords(spots, space='angular')
mask = valid.squeeze() > 0.5
obs = coords.squeeze()[mask].detach().clone()
print(f'{obs.shape[0]} valid spots')

## Gradient sanity check

In [ ]:
euler = gt_euler.clone().requires_grad_(True)
spots = model(euler.unsqueeze(0), gt_pos.unsqueeze(0), lattice_params=gt_latc)
loss = (spots.omega * spots.valid).pow(2).sum()
loss.backward()
print('Gradient on Euler angles:', euler.grad)

## Single-grain recovery from a perturbed start

In [ ]:
torch.manual_seed(42)
init_euler = gt_euler + torch.randn(3) * 3.0 * DEG2RAD
init_latc = gt_latc + torch.randn(6) * torch.tensor([5e-3, 5e-3, 5e-3, 5e-2, 5e-2, 5e-2])

# Resolution-based weights (typical FF: 2theta tight, eta loose, omega medium)
sigma_2th = 0.5 * geom.px / geom.Lsd
sigma_eta = 0.5 / 500.0
sigma_ome = 0.25 * abs(geom.omega_step) * DEG2RAD
weights = torch.tensor([sigma_2th, sigma_eta, sigma_ome])
weights = weights / weights.mean()
loss_fn = md.SpotMatchingLoss(metric='l2', weights=weights)

result = md.optimize_single_grain(
    model,
    observed_spots=obs,
    init_euler=init_euler,
    init_lattice=init_latc,
    position=gt_pos,
    loss=loss_fn,
    verbose=True,
)

In [ ]:
eval_ = md.evaluate_recovery(result, gt_euler, gt_latc)
print(f"Final misorientation: {eval_['misori_deg']:.6f} deg")
print(f"Lattice max error:    {eval_['lattice_max_err']:.2e}")
print(f"Lattice errors:        {eval_['lattice_errors'].tolist()}")

Misorientation should converge well below 0.1 deg and lattice errors well
below 1e-3 A on noise-free synthetic data, demonstrating that
`midas-diffract` is fully end-to-end differentiable through orientation
and lattice parameters.